In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install geopandas mapclassify folium -q

In [7]:
import geopandas as gpd
import pandas as pd

# Load Dataset 1 - Development Activity Model Footprints
footprints = gpd.read_file('/content/drive/MyDrive/Colab Notebooks/MOP/GreenRooftopPolygonLayers')

# Load Dataset 2 - Solar Rooftop Suitability
solar = gpd.read_file('/content/drive/MyDrive/Colab Notebooks/MOP/development-activity-model-footprints.geojson')

# Quick check
print("=== Development Activity Model Footprints ===")
print(f"Rows: {len(footprints)}")
print(f"Columns: {list(footprints.columns)}")
print(f"CRS: {footprints.crs}")
print()
print("=== Solar Rooftop Suitability ===")
print(f"Rows: {len(solar)}")
print(f"Columns: {list(solar.columns)}")
print(f"CRS: {solar.crs}")

=== Development Activity Model Footprints ===
Rows: 119530
Columns: ['Shape_Leng', 'Shape_Area', 'RATING', 'geometry']
CRS: EPSG:28355

=== Solar Rooftop Suitability ===
Rows: 1274
Columns: ['dev_key', 'status', 'permit_num', 'bldhgt_ahd', 'base_ahd', 'address', 'num_floors', 'land_use_1', 'land_use_2', 'land_use_3', 'shape_type', 'datadate', 'geo_point_2d', 'geometry']
CRS: EPSG:4326


In [8]:
# Swap the variable names - they loaded in reverse
solar_gdf = footprints.copy()    # this is actually the solar data
footprints_gdf = solar.copy()    # this is actually the footprints data

# Confirm they are correct now
print("=== Solar Rooftop - Columns ===")
print(list(solar_gdf.columns))
print(f"Rows: {len(solar_gdf)}")
print(f"CRS: {solar_gdf.crs}")
print()
print("=== Development Footprints - Columns ===")
print(list(footprints_gdf.columns))
print(f"Rows: {len(footprints_gdf)}")
print(f"CRS: {footprints_gdf.crs}")
print()

# Now check the RATING values
print("=== Solar RATING unique values ===")
print(solar_gdf['RATING'].value_counts())
print()

# Check STATUS values in footprints
print("=== Footprints STATUS unique values ===")
print(footprints_gdf['status'].value_counts())

=== Solar Rooftop - Columns ===
['Shape_Leng', 'Shape_Area', 'RATING', 'geometry']
Rows: 119530
CRS: EPSG:28355

=== Development Footprints - Columns ===
['dev_key', 'status', 'permit_num', 'bldhgt_ahd', 'base_ahd', 'address', 'num_floors', 'land_use_1', 'land_use_2', 'land_use_3', 'shape_type', 'datadate', 'geo_point_2d', 'geometry']
Rows: 1274
CRS: EPSG:4326

=== Solar RATING unique values ===
RATING
Good         49666
Moderate     22772
Poor         20883
Excellent    18195
Very Poor     8014
Name: count, dtype: int64

=== Footprints STATUS unique values ===
status
COMPLETED             554
APPROVED              450
UNDER CONSTRUCTION    139
APPLIED               131
Name: count, dtype: int64


In [9]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# STEP 1 - CLEAN SOLAR DATASET
# ============================================================

print("--- Cleaning Solar Dataset ---")

# Check for nulls
print(f"Nulls before cleaning:\n{solar_gdf.isnull().sum()}")

# Drop rows with missing RATING or geometry
solar_clean = solar_gdf.dropna(subset=['RATING', 'geometry', 'Shape_Area'])

# Remove zero or negative areas
solar_clean = solar_clean[solar_clean['Shape_Area'] > 0]

# Calculate usable rooftop area (75% efficiency factor)
solar_clean = solar_clean.copy()
solar_clean['usable_area_m2'] = solar_clean['Shape_Area'] * 0.75

# Map RATING to numeric score for calculations
rating_map = {
    'Excellent': 1.0,
    'Good':      0.75,
    'Moderate':  0.5,
    'Poor':      0.25,
    'Very Poor': 0.1
}
solar_clean['rating_score'] = solar_clean['RATING'].map(rating_map)

# Calculate estimated annual kWh
# Formula: usable_area x panel_efficiency(15%) x peak_sun_hours(4.5) x days(365)
solar_clean['est_kwh_year'] = (
    solar_clean['usable_area_m2'] *
    solar_clean['rating_score'] *
    0.15 *
    4.5 *
    365
)

print(f"\nRows after cleaning: {len(solar_clean)}")
print(f"Nulls after cleaning:\n{solar_clean.isnull().sum()}")
print(f"\nSample of cleaned solar data:")
print(solar_clean[['Shape_Area', 'usable_area_m2', 'RATING', 'rating_score', 'est_kwh_year']].head())

# ============================================================
# STEP 2 - CLEAN FOOTPRINTS DATASET
# ============================================================

print("\n--- Cleaning Footprints Dataset ---")

# Check nulls
print(f"Nulls before cleaning:\n{footprints_gdf.isnull().sum()}")

# Keep only relevant columns
cols_to_keep = ['dev_key', 'status', 'bldhgt_ahd', 'base_ahd',
                'address', 'num_floors', 'land_use_1',
                'shape_type', 'geometry']
footprints_clean = footprints_gdf[cols_to_keep].copy()

# Filter to COMPLETED and UNDER CONSTRUCTION only (most actionable)
footprints_clean = footprints_clean[
    footprints_clean['status'].isin(['COMPLETED', 'UNDER CONSTRUCTION'])
]

print(f"\nRows after filtering to COMPLETED + UNDER CONSTRUCTION: {len(footprints_clean)}")
print(f"\nStatus breakdown:\n{footprints_clean['status'].value_counts()}")

# ============================================================
# STEP 3 - ALIGN CRS BEFORE SPATIAL JOIN
# ============================================================

print("\n--- Aligning CRS ---")
print(f"Solar CRS: {solar_clean.crs}")
print(f"Footprints CRS: {footprints_clean.crs}")

# Convert footprints to match solar CRS (EPSG:28355)
footprints_projected = footprints_clean.to_crs(solar_clean.crs)
print(f"Footprints CRS after conversion: {footprints_projected.crs}")
print("\nCRS aligned successfully!")

--- Cleaning Solar Dataset ---
Nulls before cleaning:
Shape_Leng    0
Shape_Area    0
RATING        0
geometry      0
dtype: int64

Rows after cleaning: 119530
Nulls after cleaning:
Shape_Leng        0
Shape_Area        0
RATING            0
geometry          0
usable_area_m2    0
rating_score      0
est_kwh_year      0
dtype: int64

Sample of cleaned solar data:
   Shape_Area  usable_area_m2     RATING  rating_score  est_kwh_year
0        92.0            69.0  Excellent          1.00  16999.875002
1         8.0             6.0       Good          0.75   1108.687500
2        64.0            48.0       Good          0.75   8869.500001
3       100.0            75.0  Excellent          1.00  18478.124997
4         4.0             3.0       Good          0.75    554.343750

--- Cleaning Footprints Dataset ---
Nulls before cleaning:
dev_key           0
status            0
permit_num        0
bldhgt_ahd        0
base_ahd          0
address           0
num_floors        0
land_use_1       14
